# Explore ingest.py
Inspect what `load_documents()` and `chunk_documents()` produce before running the full (embedding) ingest.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src import ingest

## 1. Loaded documents (one per file in `data/`)

In [ ]:
documents = ingest.load_documents()
print(f"{len(documents)} documents loaded\n")

for doc in documents:
    print(f"source={doc.metadata['source']!r} type={doc.metadata['type']!r} "
          f"skills={doc.metadata['skills']!r} chars={len(doc.page_content)}")

## 1b. Loaded PDF cover letters (`data/cover letters/*.pdf`)

In [ ]:
pdf_documents = ingest.load_pdf_documents()
print(f"{len(pdf_documents)} PDF documents loaded\n")

for doc in pdf_documents:
    print(f"source={doc.metadata['source']!r} type={doc.metadata['type']!r} "
          f"chars={len(doc.page_content)}")

## 2. Chunked documents (token-based, 500/50 overlap)

In [ ]:
chunks = ingest.chunk_documents(documents + pdf_documents)
print(f"{len(chunks)} chunks produced\n")

from collections import Counter
counts = Counter(c.metadata["source"] for c in chunks)
for source, count in counts.items():
    print(f"{source}: {count} chunk(s)")

In [ ]:
# Peek at one chunk's content
sample = chunks[0]
print(sample.metadata)
print("---")
print(sample.page_content[:500])

## 3. (Optional) Run full ingest — embeds chunks via OpenAI and persists to `chroma_db/`
This calls the OpenAI embeddings API, so it costs a small amount and requires `OPENAI_API_KEY` in `.env`.

In [ ]:
vectorstore = ingest.ingest()